**Imports**

In [ ]:
import hashlib
import itertools
import json
import math
import time
from dataclasses import dataclass, replace
from datetime import datetime, timezone
from pathlib import Path
from typing import Callable, Iterable, Mapping

import yaml

A **cycle** is the time one bus needs to leave its starting terminal, drive the whole route out and back (stopping at each stop on the way), take its layover, and be ready to start the same trip again. We take the full round trip, not one direction, because a bus can only start its next trip after it gets back. The number of buses a route needs is therefore set by the full round trip, `vehicles = ceil(cycle / headway)`. The two directions can also have different lengths (one-way streets) and different stop counts, so doubling one direction would be wrong.


**1. Dwell is in seconds, the cycle is in minutes. Where does the conversion happen?**
Inside `cycle_min`, at one point only: `dwell_s / 60`. The caller passes dwell in its natural unit (seconds, as in the assumption sheet: 20 s), and the function returns minutes. The parameter names carry the unit (`_s`, `_min`, `_km`, `_kmh`), so mixing them up is visible.

**2. What should happen if the speed is 0 or negative?**
Raise a `ValueError`. A speed of 0 would mean infinite run time and a negative speed has no meaning. Returning `inf`, `0`, or a clamped value would hide a data error and go on to give a wrong vehicle count. That breaks the rule *fail loudly, do not guess*. The same applies to negative lengths, dwell, layover or stop counts, and to non-finite values.

**3. If only half of the route is known (partial coverage), should this function still return a number?**
No. This function does not decide that. It is pure arithmetic: it trusts that the inputs describe the **full** round trip. Whether a route is fully covered by curated stops and drive paths is decided earlier, where the route length is built (the sum of drive paths between consecutive curated stops). That step must refuse to produce a length for a partially covered route, so `cycle_min` is never called with half a route.


**Cycle time for one full round trip**

In [2]:
def cycle_min(
    route_length_km: float,
    speed_kmh: float,
    stops_per_cycle: int,
    dwell_s: float,
    layover_min: float,
) -> float:
    """Full round-trip cycle time in minutes.

    cycle = route_length_km / speed_kmh * 60 + stops_per_cycle * dwell_s / 60 + layover_min

    route_length_km: both directions together (sum of drive paths between curated stops).
    stops_per_cycle: stop events over both directions.
    layover_min: total layover per cycle.
    """
    values = {
        "route_length_km": route_length_km,
        "speed_kmh": speed_kmh,
        "stops_per_cycle": stops_per_cycle,
        "dwell_s": dwell_s,
        "layover_min": layover_min,
    }
    for name, value in values.items():
        if not math.isfinite(value):
            raise ValueError(f"{name} must be finite, got {value}")
        if value < 0:
            raise ValueError(f"{name} must be >= 0, got {value}")
    if speed_kmh == 0:
        raise ValueError("speed_kmh must be > 0, got 0")
    if route_length_km == 0:
        raise ValueError("route_length_km must be > 0, got 0")

    run_min = route_length_km / speed_kmh * 60
    dwell_min = stops_per_cycle * dwell_s / 60
    return run_min + dwell_min + layover_min


**cycle_min quick check**

In [3]:
# 12 km round trip at 15 km/h, 30 stop events x 20 s, 5 min layover
# run 48 + dwell 10 + layover 5 = 63 min
assert math.isclose(cycle_min(12, 15, 30, 20, 5), 63.0)

for bad_speed in (0, -10):
    try:
        cycle_min(12, bad_speed, 30, 20, 5)
    except ValueError as e:
        print("rejected:", e)
    else:
        raise AssertionError("bad speed accepted")

cycle_min(12, 15, 30, 20, 5)


rejected: speed_kmh must be > 0, got 0
rejected: speed_kmh must be >= 0, got -10


63.0

**cycle_min tests**

In [4]:
# --- cycle_min tests (expected values computed by hand) ---

def check(expected, **kw):
    got = cycle_min(**kw)
    assert math.isclose(got, expected, abs_tol=1e-9), f"{kw} -> {got}, expected {expected}"

def must_fail(**kw):
    try:
        cycle_min(**kw)
    except ValueError:
        return
    raise AssertionError(f"bad input accepted: {kw}")

# 12 km round trip at 15 km/h, 30 stop events x 20 s, 5 min layover
base = dict(route_length_km=12, speed_kmh=15, stops_per_cycle=30, dwell_s=20, layover_min=5)

# 1. Normal case: run 48 + dwell 10 + layover 5 = 63
check(63.0, **base)

# 2. Each part checked on its own (catches a term that is missing or counted twice)
check(53.0, **{**base, "dwell_s": 0})                              # 48 + 0 + 5
check(58.0, **{**base, "layover_min": 0})                          # 48 + 10 + 0
check(48.0, **{**base, "stops_per_cycle": 0, "layover_min": 0})    # run time only

# 3. Lower speed -> longer cycle (the stress-test direction): 60 + 10 + 5 = 75
check(75.0, **{**base, "speed_kmh": 12})

# 4. Seconds-to-minutes conversion: 6 stops x 30 s = 3 min
check(48.0 + 3.0, **{**base, "stops_per_cycle": 6, "dwell_s": 30, "layover_min": 0})

# 5. Invalid inputs must fail loudly
for name, bad in [
    ("speed_kmh", 0), ("speed_kmh", -10),
    ("route_length_km", 0), ("route_length_km", -1),
    ("dwell_s", -1), ("layover_min", -1), ("stops_per_cycle", -1),
    ("speed_kmh", float("nan")), ("route_length_km", float("inf")),
]:
    must_fail(**{**base, name: bad})

print("cycle_min: all tests passed")

cycle_min: all tests passed


**Vehicles needed for a headway**

In [ ]:
VEHICLE_RATIO_TOL = 1e-9


def vehicles(cycle_min: float, headway_min: float) -> int:
    """Vehicles needed to run one cycle at a headway: ceil(cycle / headway).

    A ratio within 1e-9 of an integer is float noise and uses that integer.
    A headway longer than the cycle needs one vehicle.
    """
    for name, value in (("cycle_min", cycle_min), ("headway_min", headway_min)):
        if not math.isfinite(value):
            raise ValueError(f"{name} must be finite, got {value}")
        if value <= 0:
            raise ValueError(f"{name} must be > 0, got {value}")
    if headway_min > cycle_min:
        return 1
    ratio = cycle_min / headway_min
    nearest = round(ratio)
    if abs(ratio - nearest) <= VEHICLE_RATIO_TOL:
        return int(nearest)
    return math.ceil(ratio)

**Model catalog (stub)**

In [ ]:
COVERAGE_VALUES = ("full", "partial")
CYCLE_FIELDS = ("full_cycle_km", "speed_kmh", "stops_per_cycle", "dwell_s", "layover_min")
LINK_KINDS = ("walk", "bike")


@dataclass(frozen=True)
class ServiceGroup:
    """Existing service group or pre-built service. Cycle facts may be None only when coverage is partial."""
    service_group_id: str
    pattern_ids: tuple[str, ...]
    base_headway_min: float
    full_cycle_km: float | None
    speed_kmh: float | None
    stops_per_cycle: int | None
    dwell_s: float | None
    layover_min: float | None
    coverage: str
    status: str
    speed_status: str

    def __post_init__(self):
        if self.coverage not in COVERAGE_VALUES:
            raise ValueError(f"{self.service_group_id}: coverage must be one of {COVERAGE_VALUES}, got {self.coverage!r}")
        if self.coverage == "full":
            missing = [name for name in CYCLE_FIELDS if getattr(self, name) is None]
            if missing:
                raise ValueError(f"{self.service_group_id}: full coverage but missing {missing}")


@dataclass(frozen=True)
class Link:
    link_id: str
    kind: str
    length_m: float
    requires_construction: bool
    status: str

    def __post_init__(self):
        if self.kind not in LINK_KINDS:
            raise ValueError(f"{self.link_id}: kind must be one of {LINK_KINDS}, got {self.kind!r}")
        if not math.isfinite(self.length_m) or self.length_m < 0:
            raise ValueError(f"{self.link_id}: length_m must be finite and >= 0, got {self.length_m}")


@dataclass(frozen=True)
class Parking:
    parking_id: str
    status: str


@dataclass(frozen=True)
class ModelCatalog:
    """Stub of Celal's ModelCatalog with the fields compute_resources needs."""
    service_groups: Mapping[str, ServiceGroup]
    prebuilt_services: Mapping[str, ServiceGroup]
    links: Mapping[str, Link]
    parking: Mapping[str, Parking]

**Intervention candidate**

In [ ]:
CANDIDATE_KINDS = ("headway_cut", "activate_service", "link_bundle")


@dataclass(frozen=True)
class Candidate:
    candidate_id: str
    kind: str
    target_id: str | None = None
    pct: float | None = None
    link_ids: tuple[str, ...] = ()
    parking_ids: tuple[str, ...] = ()

    def __post_init__(self):
        cid = self.candidate_id
        if self.kind not in CANDIDATE_KINDS:
            raise ValueError(f"{cid}: kind must be one of {CANDIDATE_KINDS}, got {self.kind!r}")
        if self.kind == "link_bundle":
            if self.target_id is not None or self.pct is not None:
                raise ValueError(f"{cid}: link_bundle takes no target_id or pct")
            if not self.link_ids and not self.parking_ids:
                raise ValueError(f"{cid}: link_bundle needs link_ids or parking_ids")
            return
        if self.target_id is None:
            raise ValueError(f"{cid}: {self.kind} needs target_id")
        if self.link_ids or self.parking_ids:
            raise ValueError(f"{cid}: {self.kind} takes no link_ids or parking_ids")
        if self.kind == "activate_service":
            if self.pct is not None:
                raise ValueError(f"{cid}: activate_service takes no pct")
            return
        if self.pct is None or not math.isfinite(self.pct) or not 0 < self.pct < 1:
            raise ValueError(f"{cid}: headway_cut pct must be in (0, 1), got {self.pct}")

**Scenario with analysis hours from limits.yaml**

In [ ]:
@dataclass(frozen=True)
class Scenario:
    speed_factor: float
    headway_factor: float
    H_hours: float
    headway_factor_by_group: Mapping[str, float] | None = None  # if given: one factor per existing group, no fallback

    def __post_init__(self):
        for name in ("speed_factor", "headway_factor", "H_hours"):
            value = getattr(self, name)
            if value is None:
                raise ValueError(f"{name} is missing")
            if isinstance(value, bool):
                raise ValueError(f"{name} must be a number, got bool {value}")
            if not math.isfinite(value) or value <= 0:
                raise ValueError(f"{name} must be finite and > 0, got {value}")
        for group, value in (self.headway_factor_by_group or {}).items():
            if isinstance(value, bool) or not math.isfinite(value) or value <= 0:
                raise ValueError(f"headway factor for {group} must be finite and > 0, got {value!r}")

    def group_headway_factor(self, group_id: str) -> float:
        if self.headway_factor_by_group is None:
            return self.headway_factor
        if group_id not in self.headway_factor_by_group:
            raise ValueError(f"headway_factor_by_group has no factor for service group {group_id}")
        return self.headway_factor_by_group[group_id]


def load_scenario(limits_path: str | Path, speed_factor: float, headway_factor: float) -> Scenario:
    """Scenario with H_hours taken from config/limits.yaml."""
    with open(limits_path) as f:
        limits = yaml.safe_load(f) or {}
    if "H_hours" not in limits:
        raise ValueError(f"H_hours missing in {limits_path}")
    return Scenario(speed_factor=speed_factor, headway_factor=headway_factor, H_hours=limits["H_hours"])

**Resource result type**

In [ ]:
@dataclass(frozen=True)
class ServiceResource:
    service_id: str
    candidate_id: str
    base_headway_min: float | None
    new_headway_min: float
    base_cycle_min: float | None
    new_cycle_min: float | None
    base_vehicles: int | None
    new_vehicles: int | None
    delta_vehicles: int | None
    status: str
    warnings: tuple[str, ...]


@dataclass(frozen=True)
class ResourceResult:
    added_bus_hours: float | None
    known_part_bus_hours: float
    new_infra_km: float
    action_count: int
    status: str
    warnings: tuple[str, ...]
    per_service: tuple[ServiceResource, ...]

**Resources for one package**

In [ ]:
M_PER_KM = 1000


def _unique_candidates(package: Iterable[Candidate]) -> list[Candidate]:
    by_id: dict[str, Candidate] = {}
    for c in package:
        if c.candidate_id in by_id and by_id[c.candidate_id] != c:
            raise ValueError(f"{c.candidate_id}: two different definitions in one package")
        by_id[c.candidate_id] = c
    return [by_id[cid] for cid in sorted(by_id)]


def _cycle(service: ServiceGroup, scenario: Scenario) -> float:
    return cycle_min(
        service.full_cycle_km,
        service.speed_kmh * scenario.speed_factor,
        service.stops_per_cycle,
        service.dwell_s,
        service.layover_min,
    )


def _headway_cut(cut: Candidate, group: ServiceGroup, scenario: Scenario) -> ServiceResource:
    base_headway = group.base_headway_min * scenario.group_headway_factor(group.service_group_id)
    new_headway = base_headway * (1 - cut.pct)
    if group.coverage == "partial":
        return ServiceResource(group.service_group_id, cut.candidate_id, base_headway, new_headway,
                               None, None, None, None, None, "unknown", ())
    cycle = _cycle(group, scenario)
    base_v = vehicles(cycle, base_headway)
    new_v = vehicles(cycle, new_headway)
    delta = new_v - base_v
    if delta < 0:
        raise ValueError(f"{cut.candidate_id}: negative vehicle delta {delta} on {group.service_group_id}")
    warnings = []
    if base_headway > cycle or new_headway > cycle:
        warnings.append("headway_exceeds_cycle")
    if delta == 0:
        warnings.append("zero_cost_uses_slack")
    return ServiceResource(group.service_group_id, cut.candidate_id, base_headway, new_headway,
                           cycle, cycle, base_v, new_v, delta, "known", tuple(warnings))


def _activation(act: Candidate, service: ServiceGroup, scenario: Scenario) -> ServiceResource:
    # Pre-built service headway is a design value; headway_factor applies to existing base headways only
    headway = service.base_headway_min
    if service.coverage == "partial":
        return ServiceResource(service.service_group_id, act.candidate_id, None, headway,
                               None, None, 0, None, None, "unknown", ())
    cycle = _cycle(service, scenario)
    new_v = vehicles(cycle, headway)
    warnings = ("headway_exceeds_cycle",) if headway > cycle else ()
    return ServiceResource(service.service_group_id, act.candidate_id, None, headway,
                           None, cycle, 0, new_v, new_v, "known", warnings)


def compute_resources(package: Iterable[Candidate], catalog: ModelCatalog, scenario: Scenario) -> ResourceResult:
    """Resources one package needs under one scenario. Does not compare against limits."""
    if scenario.headway_factor_by_group is not None:
        missing = sorted(set(catalog.service_groups) - set(scenario.headway_factor_by_group))
        if missing:
            raise ValueError(f"headway_factor_by_group has no factor for service groups {missing}")
        # pre-built services keep their design headway; only speed_factor applies to them
        not_groups = sorted(set(scenario.headway_factor_by_group) - set(catalog.service_groups))
        if not_groups:
            raise ValueError(f"headway_factor_by_group has ids that are not existing service groups: {not_groups}")
    candidates = _unique_candidates(package)
    cuts: dict[str, Candidate] = {}
    activations: dict[str, Candidate] = {}
    link_ids: set[str] = set()

    for c in candidates:
        if c.kind == "headway_cut":
            if c.target_id not in catalog.service_groups:
                raise ValueError(f"{c.candidate_id}: unknown service group {c.target_id!r}")
            if c.target_id in cuts:
                raise ValueError(f"two headway actions on {c.target_id}: {cuts[c.target_id].candidate_id}, {c.candidate_id}")
            cuts[c.target_id] = c
        elif c.kind == "activate_service":
            if c.target_id not in catalog.prebuilt_services:
                raise ValueError(f"{c.candidate_id}: unknown pre-built service {c.target_id!r}")
            if c.target_id in activations:
                raise ValueError(f"two activations of {c.target_id}: {activations[c.target_id].candidate_id}, {c.candidate_id}")
            activations[c.target_id] = c
        else:
            for link_id in c.link_ids:
                if link_id not in catalog.links:
                    raise ValueError(f"{c.candidate_id}: unknown link {link_id!r}")
                link_ids.add(link_id)
            for parking_id in c.parking_ids:
                if parking_id not in catalog.parking:
                    raise ValueError(f"{c.candidate_id}: unknown parking {parking_id!r}")

    computed = [_headway_cut(cut, catalog.service_groups[sid], scenario) for sid, cut in cuts.items()]
    computed += [_activation(act, catalog.prebuilt_services[sid], scenario) for sid, act in activations.items()]
    per_service = tuple(sorted(computed, key=lambda s: s.service_id))
    warnings = tuple(dict.fromkeys(w for s in per_service for w in s.warnings))
    known_part = sum(s.delta_vehicles for s in per_service if s.status == "known") * scenario.H_hours
    status = "unknown" if any(s.status == "unknown" for s in per_service) else "known"
    construction_m = sum(catalog.links[lid].length_m for lid in link_ids if catalog.links[lid].requires_construction)

    return ResourceResult(
        added_bus_hours=None if status == "unknown" else known_part,
        known_part_bus_hours=known_part,
        new_infra_km=construction_m / M_PER_KM,
        action_count=len(candidates),
        status=status,
        warnings=warnings,
        per_service=per_service,
    )

**Routing request and evaluation result (stubs)**

In [ ]:
# STUB: replace with Celal's contracts (RoutingRequest, EvaluationResult) when they land
OD_STATUSES = ("reachable", "over_cap", "no_path")


@dataclass(frozen=True)
class RoutingRequest:
    """Stub. Package ids, accessibility thresholds T and time cap C, all in minutes."""
    candidate_ids: tuple[str, ...]
    thresholds_min: tuple[int, ...]
    primary_t_min: int
    cap_min: float

    def __post_init__(self):
        if self.primary_t_min not in self.thresholds_min:
            raise ValueError(f"primary_t_min {self.primary_t_min} not in thresholds {self.thresholds_min}")
        if isinstance(self.cap_min, bool) or not math.isfinite(self.cap_min) or self.cap_min <= 0:
            raise ValueError(f"cap_min must be finite and > 0, got {self.cap_min}")


@dataclass(frozen=True)
class ODResult:
    """Stub. Unreachable ODs carry minutes = C and status no_path, never NaN or inf."""
    od_id: str
    minutes: float
    status: str

    def __post_init__(self):
        if self.status not in OD_STATUSES:
            raise ValueError(f"{self.od_id}: status must be one of {OD_STATUSES}, got {self.status!r}")
        if not math.isfinite(self.minutes):
            raise ValueError(f"{self.od_id}: minutes must be finite, got {self.minutes}")


@dataclass(frozen=True)
class EvaluationResult:
    """Stub. within_counts maps each threshold T to the number of ODs reached within T."""
    config_hash: str
    candidate_ids: tuple[str, ...]
    od_results: tuple[ODResult, ...]
    within_counts: Mapping[int, int]
    primary_t_min: int
    capped_time_sum: float

**Fixture evaluator**

In [ ]:
FIXTURE_LABEL = "synthetic test fixture, not a city result"


def load_fixture(path: str | Path) -> dict:
    with open(path) as f:
        fixture = json.load(f)
    if fixture.get("label") != FIXTURE_LABEL:
        raise ValueError(f"{path}: missing fixture label {FIXTURE_LABEL!r}")
    return fixture


def make_fixture_evaluator(fixture: Mapping) -> Callable[[RoutingRequest], EvaluationResult]:
    """evaluator(RoutingRequest) -> EvaluationResult over a synthetic fixture.

    OD time = min(disrupted time, every option whose required actions are all in the package).
    """
    action_ids = set(fixture["actions"])
    config_hash = hashlib.sha256(json.dumps(fixture, sort_keys=True).encode()).hexdigest()[:12]

    def evaluate(request: RoutingRequest) -> EvaluationResult:
        package = set(request.candidate_ids)
        unknown = package - action_ids
        if unknown:
            raise ValueError(f"unknown candidate ids {sorted(unknown)}")
        od_results = []
        for od in fixture["ods"]:
            times = [] if od["disrupted_min"] is None else [od["disrupted_min"]]
            times += [opt["minutes"] for opt in od["options"] if set(opt["requires"]) <= package]
            if not times:
                od_results.append(ODResult(od["od_id"], request.cap_min, "no_path"))
                continue
            t = min(times)
            od_results.append(ODResult(od["od_id"], t, "over_cap" if t > request.cap_min else "reachable"))
        within = {
            t_min: sum(1 for r in od_results if r.status != "no_path" and r.minutes <= t_min)
            for t_min in request.thresholds_min
        }
        capped = sum(min(r.minutes, request.cap_min) for r in od_results)
        return EvaluationResult(config_hash, tuple(sorted(package)), tuple(od_results),
                                within, request.primary_t_min, capped)

    return evaluate

**Ranking key for packages (count or sum objective)**

In [ ]:
OBJECTIVES = ("count", "sum")


def rank_key(result: EvaluationResult, resources: ResourceResult, objective: str) -> tuple:
    """Sort key, best first. count: more ODs within T, then lower capped sum; sum: the reverse.
    Then fewer bus-hours, fewer new km, fewer actions, candidate ids alphabetically."""
    if objective not in OBJECTIVES:
        raise ValueError(f"objective must be one of {OBJECTIVES}, got {objective!r}")
    if resources.added_bus_hours is None:
        raise ValueError(f"cannot rank {result.candidate_ids}: bus-hours unknown")
    within = -result.within_counts[result.primary_t_min]
    primary = (within, result.capped_time_sum) if objective == "count" else (result.capped_time_sum, within)
    return (
        *primary,
        resources.added_bus_hours,
        resources.new_infra_km,
        resources.action_count,
        tuple(sorted(result.candidate_ids)),
    )

**Resource limits from limits.yaml**

In [ ]:
LIMIT_FIELDS = ("B_h_bus_hours", "K_new_infra_km", "N_a_actions", "max_exact_packages")


@dataclass(frozen=True)
class Limits:
    B_h_bus_hours: float
    K_new_infra_km: float
    N_a_actions: int
    max_exact_packages: int

    def __post_init__(self):
        for name in LIMIT_FIELDS:
            value = getattr(self, name)
            if value is None or isinstance(value, bool) or not math.isfinite(value) or value < 0:
                raise ValueError(f"{name} must be a finite number >= 0, got {value!r}")


def load_limits(path: str | Path) -> Limits:
    with open(path) as f:
        raw = yaml.safe_load(f) or {}
    missing = [name for name in LIMIT_FIELDS if name not in raw]
    if missing:
        raise ValueError(f"{path}: missing {missing}")
    return Limits(**{name: raw[name] for name in LIMIT_FIELDS})

**Gate: choose and commit the objective**

In [ ]:
GATE_THRESHOLDS_MIN = (40, 45, 50)
GATE_MIN_LOST = 20
GATE_LOW_SHARE = 0.15  # spec v4.1 says 0.10; 0.15 is a simulation-backed team decision
GATE_HIGH_SHARE = 0.90


def _within(od: ODResult, t_min: float) -> bool:
    return od.status != "no_path" and od.minutes <= t_min


def count_lost(normal: EvaluationResult, disrupted: EvaluationResult, thresholds_min=GATE_THRESHOLDS_MIN) -> dict[int, int]:
    """lost_T = ODs within T in normal and not within T when disrupted (no_path counts as > T)."""
    normal_by_id = {od.od_id: od for od in normal.od_results}
    disrupted_by_id = {od.od_id: od for od in disrupted.od_results}
    if normal_by_id.keys() != disrupted_by_id.keys():
        raise ValueError("normal and disrupted results cover different ODs")
    return {
        t: sum(1 for od_id, n in normal_by_id.items() if _within(n, t) and not _within(disrupted_by_id[od_id], t))
        for t in thresholds_min
    }


def gate_objective(lost_by_t: Mapping[int, int], total_od: int, thresholds_min=GATE_THRESHOLDS_MIN) -> tuple[str, dict[int, float]]:
    """"sum" if every T has few lost ODs or an extreme lost share, else "count"."""
    if total_od == 0:
        raise ValueError("total_OD is 0: no ODs to evaluate")
    missing = [t for t in thresholds_min if t not in lost_by_t]
    if missing:
        raise ValueError(f"lost counts missing for T={missing}")
    share = {t: lost_by_t[t] / total_od for t in thresholds_min}
    quiet = all(
        lost_by_t[t] < GATE_MIN_LOST or share[t] < GATE_LOW_SHARE or share[t] > GATE_HIGH_SHARE
        for t in thresholds_min
    )
    return ("sum" if quiet else "count"), share


def gate_decision(normal: EvaluationResult, disrupted: EvaluationResult, path: str | Path) -> dict:
    """Decide the objective once and commit it to objective_decision.json. Never overwrites."""
    path = Path(path)
    if path.exists():
        raise FileExistsError(f"{path}: objective already committed")
    total_od = len(normal.od_results)
    lost = count_lost(normal, disrupted)
    objective, share = gate_objective(lost, total_od)
    record = {
        "objective": objective,
        "lost_T": {str(t): lost[t] for t in GATE_THRESHOLDS_MIN},
        "share_T": {str(t): share[t] for t in GATE_THRESHOLDS_MIN},
        "total_OD": total_od,
        "thresholds": {
            "T_min": list(GATE_THRESHOLDS_MIN),
            "min_lost": GATE_MIN_LOST,
            "low_share": GATE_LOW_SHARE,
            "high_share": GATE_HIGH_SHARE,
        },
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "input_hashes": {"normal": normal.config_hash, "disrupted": disrupted.config_hash},
    }
    path.write_text(json.dumps(record, indent=2))
    return record


def load_objective(path: str | Path) -> str:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"{path}: objective decision missing; run gate_decision first")
    objective = json.loads(path.read_text()).get("objective")
    if objective not in OBJECTIVES:
        raise ValueError(f"{path}: objective must be one of {OBJECTIVES}, got {objective!r}")
    return objective

**Decision problem and production resources**

In [ ]:
@dataclass(frozen=True)
class DecisionProblem:
    """Candidates, rules and limits for one optimisation. resources(ids) is injected like the evaluator."""
    candidate_ids: tuple[str, ...]
    exclusions: tuple[tuple[str, ...], ...]
    prerequisites: Mapping[str, tuple[str, ...]]
    distance_m: Mapping[str, float]
    limits: Limits
    thresholds_min: tuple[int, ...]
    primary_t_min: int
    cap_min: float
    objective_path: Path
    resources: Callable[[tuple[str, ...]], ResourceResult]

    def __post_init__(self):
        known = set(self.candidate_ids)
        referenced = {i for group in self.exclusions for i in group}
        referenced |= set(self.prerequisites) | {p for ps in self.prerequisites.values() for p in ps}
        if referenced - known:
            raise ValueError(f"unknown candidate ids in exclusions/prerequisites: {sorted(referenced - known)}")
        missing = known - set(self.distance_m)
        if missing:
            raise ValueError(f"distance_m missing for {sorted(missing)}")
        if self.primary_t_min not in self.thresholds_min:
            raise ValueError(f"primary_t_min {self.primary_t_min} not in thresholds {self.thresholds_min}")


def make_catalog_resources(candidates: Mapping[str, Candidate], catalog: ModelCatalog,
                           scenario: Scenario) -> Callable[[tuple[str, ...]], ResourceResult]:
    """Production resources(ids): compute_resources and nothing else."""
    def resources(ids: tuple[str, ...]) -> ResourceResult:
        return compute_resources([candidates[i] for i in ids], catalog, scenario)
    return resources

**Optimisation result types**

In [ ]:
@dataclass(frozen=True)
class PackageOutcome:
    candidate_ids: tuple[str, ...]
    evaluation: EvaluationResult | None
    resources: ResourceResult
    feasible: bool
    reason: str = ""

    @property
    def within_t(self) -> int:
        return self.evaluation.within_counts[self.evaluation.primary_t_min]

    @property
    def capped_time_sum(self) -> float:
        return self.evaluation.capped_time_sum


@dataclass(frozen=True)
class BaselineStep:
    candidate_id: str
    score: float | None
    outcome: str  # added | excluded | does_not_fit | no_gain | infeasible | already_included | unknown_resources


@dataclass(frozen=True)
class BaselineResult:
    name: str
    label: str
    package: PackageOutcome
    steps: tuple[BaselineStep, ...]


@dataclass(frozen=True)
class OptimizationResult:
    objective: str
    label: str  # exact | approximate
    recommendation: PackageOutcome
    recommendation_source: str  # "exact" or the baseline name
    alternatives: tuple[PackageOutcome, ...]
    baselines: Mapping[str, BaselineResult]
    not_ranked: tuple[PackageOutcome, ...]
    feasible_count: int | None
    enumerated_count: int
    approximate_reason: str | None
    runtime_s: float
    all_tied_on_primary: bool | None = None  # every ranked package shares the recommendation's primary value

**Valid packages and cached outcomes**

In [ ]:
FEASIBILITY_TOL = 1e-9


def with_prerequisites(problem: DecisionProblem, ids: Iterable[str]) -> tuple[str, ...]:
    closed, todo = set(), list(ids)
    while todo:
        cid = todo.pop()
        if cid not in closed:
            closed.add(cid)
            todo.extend(problem.prerequisites.get(cid, ()))
    return tuple(sorted(closed))


def violates_exclusions(problem: DecisionProblem, ids: Iterable[str]) -> bool:
    ids = set(ids)
    return any(len(ids & set(group)) > 1 for group in problem.exclusions)


def is_feasible(resources: ResourceResult, limits: Limits) -> bool:
    return (
        resources.status == "known"
        and resources.added_bus_hours <= limits.B_h_bus_hours + FEASIBILITY_TOL
        and resources.new_infra_km <= limits.K_new_infra_km + FEASIBILITY_TOL
        and resources.action_count <= limits.N_a_actions
    )


def enumerate_packages(problem: DecisionProblem) -> Iterable[tuple[str, ...]]:
    """Every package with <= N_a actions that respects exclusions and includes its prerequisites."""
    ids = sorted(problem.candidate_ids)
    for size in range(int(problem.limits.N_a_actions) + 1):
        for package in itertools.combinations(ids, size):
            if violates_exclusions(problem, package):
                continue
            if with_prerequisites(problem, package) != package:
                continue
            yield package


def make_outcome(problem: DecisionProblem, evaluator: Callable[[RoutingRequest], EvaluationResult]):
    """Cached package -> PackageOutcome. Packages with unknown resources are not evaluated."""
    cache: dict[tuple[str, ...], PackageOutcome] = {}

    def outcome(ids: Iterable[str]) -> PackageOutcome:
        key = tuple(sorted(set(ids)))
        if key not in cache:
            res = problem.resources(key)
            if res.status != "known":
                unknown = [s.service_id for s in res.per_service if s.status != "known"]
                cache[key] = PackageOutcome(key, None, res, False, f"resources unknown (partial coverage: {', '.join(unknown)})")
            else:
                request = RoutingRequest(key, problem.thresholds_min, problem.primary_t_min, problem.cap_min)
                cache[key] = PackageOutcome(key, evaluator(request), res, is_feasible(res, problem.limits))
        return cache[key]

    return outcome

**Baselines: no action, simple greedy, nearest gap**

In [ ]:
def _limit_share(used: float, limit: float) -> float:
    if limit == 0:
        return math.inf if used > 0 else 0.0
    return used / limit


def _try_add(problem: DecisionProblem, outcome, current: tuple[str, ...], cid: str) -> tuple[str, PackageOutcome | None]:
    if cid in current:
        return "already_included", None
    candidate = with_prerequisites(problem, (*current, cid))
    if violates_exclusions(problem, candidate):
        return "excluded", None
    result = outcome(candidate)
    if result.resources.status != "known":
        return "unknown_resources", None
    if not result.feasible:
        return "does_not_fit", None
    return "added", result


def no_action_baseline(outcome) -> BaselineResult:
    return BaselineResult("no_action", "no action", outcome(()), ())


def greedy_baseline(problem: DecisionProblem, objective: str, outcome) -> BaselineResult:
    """Simple greedy on solo gain: each action (with its prerequisites) scored alone against no action."""
    empty = outcome(())
    limits = problem.limits
    scored = []
    for cid in problem.candidate_ids:
        solo = outcome(with_prerequisites(problem, (cid,)))
        if solo.resources.status != "known":
            scored.append((cid, None, None))
            continue
        if objective == "count":
            gain = solo.within_t - empty.within_t
        else:
            gain = empty.capped_time_sum - solo.capped_time_sum
        cost = max(
            _limit_share(solo.resources.added_bus_hours, limits.B_h_bus_hours),
            _limit_share(solo.resources.new_infra_km, limits.K_new_infra_km),
            _limit_share(solo.resources.action_count, limits.N_a_actions),
        )
        scored.append((cid, gain, None if math.isinf(cost) else gain / cost))
    scored.sort(key=lambda s: (s[2] is None, -(s[2] or 0.0), s[0]))

    current, package, steps = (), empty, []
    for cid, gain, score in scored:
        if gain is None:
            outcome_name = "unknown_resources"
        elif gain <= 0:
            outcome_name = "no_gain"
        elif score is None:
            outcome_name = "infeasible"
        else:
            outcome_name, added = _try_add(problem, outcome, current, cid)
            if added is not None:
                current, package = added.candidate_ids, added
        steps.append(BaselineStep(cid, score, outcome_name))
    return BaselineResult("greedy", "simple greedy (solo gain)", package, tuple(steps))


def nearest_gap_baseline(problem: DecisionProblem, outcome) -> BaselineResult:
    """Adds actions nearest to the closed link first.
    distance_m is a straight line used only for this ordering, never for routing."""
    current, package, steps = (), outcome(()), []
    for cid in sorted(problem.candidate_ids, key=lambda c: (problem.distance_m[c], c)):
        outcome_name, added = _try_add(problem, outcome, current, cid)
        if added is not None:
            current, package = added.candidate_ids, added
        steps.append(BaselineStep(cid, problem.distance_m[cid], outcome_name))
    return BaselineResult("nearest_gap", "nearest gap (straight-line ordering)", package, tuple(steps))

**Exact search with alternatives**

In [ ]:
MAX_ALTERNATIVES = 3
BASELINE_ORDER = ("greedy", "nearest_gap", "no_action")


def _fmt(value: float) -> str:
    return f"{round(value, 6):g}"


def worse_reason(alt: PackageOutcome, rec: PackageOutcome, objective: str) -> str:
    """The first rank_key element where alt is worse than rec."""
    t_min = rec.evaluation.primary_t_min
    measures = {
        "count": f"{rec.within_t - alt.within_t} fewer ODs within {t_min} min",
        "sum": f"capped time +{_fmt(alt.capped_time_sum - rec.capped_time_sum)} min",
        "bus_hours": f"+{_fmt(alt.resources.added_bus_hours - rec.resources.added_bus_hours)} bus-hours",
        "km": f"+{_fmt(alt.resources.new_infra_km - rec.resources.new_infra_km)} km new infrastructure",
        "actions": f"+{alt.resources.action_count - rec.resources.action_count} actions",
        "ids": "same measures; later by candidate id",
    }
    order = ("count", "sum") if objective == "count" else ("sum", "count")
    names = (*order, "bus_hours", "km", "actions", "ids")
    alt_key = rank_key(alt.evaluation, alt.resources, objective)
    rec_key = rank_key(rec.evaluation, rec.resources, objective)
    for name, a, r in zip(names, alt_key, rec_key):
        if a != r:
            return measures[name]
    return measures["ids"]


def optimize(problem: DecisionProblem, evaluator: Callable[[RoutingRequest], EvaluationResult]) -> OptimizationResult:
    """Exact search over all valid packages with <= N_a actions, plus baselines and alternatives.
    The objective comes only from the committed objective_decision.json."""
    started = time.perf_counter()
    objective = load_objective(problem.objective_path)
    outcome = make_outcome(problem, evaluator)

    def key(o: PackageOutcome) -> tuple:
        return rank_key(o.evaluation, o.resources, objective)

    baselines = {
        "greedy": greedy_baseline(problem, objective, outcome),
        "nearest_gap": nearest_gap_baseline(problem, outcome),
        "no_action": no_action_baseline(outcome),
    }
    packages = list(enumerate_packages(problem))
    max_exact = problem.limits.max_exact_packages

    if len(packages) > max_exact:
        source = min(BASELINE_ORDER, key=lambda name: key(baselines[name].package))
        return OptimizationResult(
            objective, "approximate", baselines[source].package, source, (), baselines, (), None,
            len(packages), f"catalog too large for exact search ({len(packages)} > {max_exact})",
            time.perf_counter() - started, None,
        )

    ranked, not_ranked = [], []
    for package in packages:
        resources = problem.resources(package)
        if resources.status != "known":
            not_ranked.append(outcome(package))
        elif is_feasible(resources, problem.limits):
            ranked.append(outcome(package))
    ranked.sort(key=key)
    recommendation, rest = ranked[0], ranked[1:]

    rec_primary = key(recommendation)[0]
    tied = [o for o in rest if key(o)[0] == rec_primary]
    shown = tied if len(tied) > MAX_ALTERNATIVES else rest[:MAX_ALTERNATIVES]
    alternatives = tuple(
        PackageOutcome(o.candidate_ids, o.evaluation, o.resources, o.feasible, worse_reason(o, recommendation, objective))
        for o in shown
    )
    return OptimizationResult(
        objective, "exact", recommendation, "exact", alternatives, baselines, tuple(not_ranked),
        len(ranked), len(packages), None, time.perf_counter() - started, len(tied) == len(rest),
    )

**Stress grid, scenario-dependent problems and stress types**

In [ ]:
STRESS_LEVELS = ("low", "default", "high")


def _check_levels(name: str, levels: Mapping[str, float]) -> None:
    if set(levels) != set(STRESS_LEVELS):
        raise ValueError(f"{name}: levels must be {STRESS_LEVELS}, got {sorted(levels)}")
    for level, value in levels.items():
        if isinstance(value, bool) or not math.isfinite(value) or value <= 0:
            raise ValueError(f"{name} {level}: must be finite and > 0, got {value!r}")


@dataclass(frozen=True)
class StressGrid:
    """3 speed levels x 3 headway levels from the assumption ranges.
    speed_factor = speed level / default speed level (all routes);
    each existing group's headway factor = its own level value / its default."""
    speed_levels_kmh: Mapping[str, float]
    headway_levels_min: Mapping[str, Mapping[str, float]]

    def __post_init__(self):
        _check_levels("speed", self.speed_levels_kmh)
        for group, levels in self.headway_levels_min.items():
            _check_levels(f"headway {group}", levels)

    def points(self, H_hours: float) -> list[tuple[str, str, Scenario]]:
        out = []
        for speed_level in STRESS_LEVELS:
            speed_factor = self.speed_levels_kmh[speed_level] / self.speed_levels_kmh["default"]
            for headway_level in STRESS_LEVELS:
                factors = {g: lv[headway_level] / lv["default"] for g, lv in self.headway_levels_min.items()}
                out.append((speed_level, headway_level, Scenario(speed_factor, 1.0, H_hours, factors)))
        return out


def stress_grid_from_assumptions(rows: Iterable[Mapping], speed_param: str, headway_params: Mapping[str, str]) -> StressGrid:
    """Grid from assumptions rows (e.g. P3 speed, P1/P2 headways per service group)."""
    by_id = {r["param_id"]: r for r in rows}

    def levels(pid: str) -> dict[str, float]:
        if pid not in by_id:
            raise ValueError(f"{pid} missing in assumptions")
        r = by_id[pid]
        return {"low": float(r["value_low"]), "default": float(r["value_default"]), "high": float(r["value_high"])}

    return StressGrid(levels(speed_param), {group: levels(pid) for group, pid in headway_params.items()})


def make_problem_for(template: DecisionProblem, candidates: Mapping[str, Candidate],
                     catalog: ModelCatalog) -> Callable[[Scenario], DecisionProblem]:
    """Production problem_for: same rules and limits, resources from compute_resources under each scenario."""
    def problem_for(scenario: Scenario) -> DecisionProblem:
        return replace(template, resources=make_catalog_resources(candidates, catalog, scenario))
    return problem_for


@dataclass(frozen=True)
class StressSetup:
    problem_for: Callable[[Scenario], DecisionProblem]
    evaluator_for: Callable[[Scenario], Callable[[RoutingRequest], EvaluationResult]]
    grid: StressGrid
    H_hours: float


@dataclass(frozen=True)
class StressPoint:
    speed_level: str
    headway_level: str
    scenario: Scenario
    feasible: bool
    bus_hours: float
    overrun: float
    new_km: float
    actions: int
    within_t: int
    capped_time_sum: float
    warnings: tuple[str, ...]


@dataclass(frozen=True)
class StressResult:
    objective: str
    package_ids: tuple[str, ...]
    points: tuple[StressPoint, ...]
    worst: StressPoint
    reoptimization: OptimizationResult
    reoptimization_reason: str
    warnings: tuple[str, ...]


def check_package(problem: DecisionProblem, package: tuple[str, ...]) -> None:
    unknown = set(package) - set(problem.candidate_ids)
    if unknown:
        raise ValueError(f"unknown candidate ids {sorted(unknown)}")
    if violates_exclusions(problem, package):
        raise ValueError(f"package {package} violates exclusions")
    if with_prerequisites(problem, package) != package:
        raise ValueError(f"package {package} misses prerequisites")

**Stress test with worst point and one re-optimisation**

In [ ]:
def _package_name(ids: tuple[str, ...]) -> str:
    return "+".join(ids) or "no action"


def _cost_rises_with_speed(points: list[StressPoint]) -> list[str]:
    """Ceil effect: at one headway level a faster point needs more bus-hours than a slower one."""
    warnings = []
    for headway_level in STRESS_LEVELS:
        by_speed = [p for p in points if p.headway_level == headway_level]
        for slow, fast in itertools.combinations(by_speed, 2):
            if fast.bus_hours > slow.bus_hours:
                warnings.append(f"cost_rises_with_speed: headway {headway_level}, speed {slow.speed_level} -> "
                                f"{fast.speed_level}: {_fmt(slow.bus_hours)} -> {_fmt(fast.bus_hours)} bus-hours")
    return warnings


def _reoptimization_reason(package: tuple[str, ...], worst: StressPoint, reopt: OptimizationResult, t_min: int) -> str:
    fit = "still fits" if worst.feasible else "does not fit"
    parts = [f"{_package_name(package)} {fit} at the worst point"]
    rec = reopt.recommendation
    if reopt.label == "approximate":
        parts.append(f"approximate: {reopt.approximate_reason}")
    elif reopt.all_tied_on_primary and reopt.objective == "count":
        parts.append(f"every feasible package reaches {rec.within_t} ODs within {t_min} min, so the capped sum decides")
    elif reopt.all_tied_on_primary:
        parts.append(f"every feasible package has capped sum {_fmt(rec.capped_time_sum)}, so ODs within {t_min} min decide")
    else:
        parts.append(f"{_package_name(rec.candidate_ids)} ranks first by {reopt.objective}")
    return "; ".join(parts)


def stress_test(setup: StressSetup, package_ids: Iterable[str]) -> StressResult:
    """Re-computes resources and journeys of one package at every grid point, then one exact
    re-optimisation at the worst point. Uses the committed objective; never runs the gate."""
    package = tuple(sorted(set(package_ids)))
    objective, t_min, points = None, None, []
    for speed_level, headway_level, scenario in setup.grid.points(setup.H_hours):
        problem = setup.problem_for(scenario)
        if objective is None:
            objective, t_min = load_objective(problem.objective_path), problem.primary_t_min
            check_package(problem, package)
        result = make_outcome(problem, setup.evaluator_for(scenario))(package)
        if result.evaluation is None:
            raise ValueError(f"{speed_level}/{headway_level}: {result.reason}")
        bus_hours = result.resources.added_bus_hours
        points.append(StressPoint(
            speed_level, headway_level, scenario, result.feasible, bus_hours,
            max(0, bus_hours - problem.limits.B_h_bus_hours), result.resources.new_infra_km,
            result.resources.action_count, result.within_t, result.capped_time_sum, result.resources.warnings,
        ))
    # largest overrun, then fewest ODs within T, then largest capped sum, then grid order
    worst = min(enumerate(points), key=lambda ip: (-ip[1].overrun, ip[1].within_t, -ip[1].capped_time_sum, ip[0]))[1]
    reopt = optimize(setup.problem_for(worst.scenario), setup.evaluator_for(worst.scenario))
    return StressResult(
        objective, package, tuple(points), worst, reopt,
        _reoptimization_reason(package, worst, reopt, t_min), tuple(_cost_rises_with_speed(points)),
    )

**Remove one action (and what depends on it)**

In [ ]:
NON_ADDITIVE_NOTE = ("losses of single removals are not additive; "
                     "do not show them as each action's share of the package")


@dataclass(frozen=True)
class RemovalResult:
    objective: str
    removed: str
    also_removed: tuple[str, ...]
    messages: tuple[str, ...]
    full: PackageOutcome
    remaining: PackageOutcome
    loss_within_t: int
    capped_time_increase: float
    note: str


def remove_action(problem: DecisionProblem, package_ids: Iterable[str], action_id: str,
                  evaluator: Callable[[RoutingRequest], EvaluationResult]) -> RemovalResult:
    """Removes one action and every action in the package that needs it as a prerequisite."""
    objective = load_objective(problem.objective_path)
    package = tuple(sorted(set(package_ids)))
    if action_id not in package:
        raise ValueError(f"{action_id} is not in package {package}")
    check_package(problem, package)
    dependents = tuple(c for c in package if c != action_id and action_id in with_prerequisites(problem, (c,)))
    remaining = tuple(c for c in package if c != action_id and c not in dependents)
    outcome = make_outcome(problem, evaluator)
    full, rest = outcome(package), outcome(remaining)
    for o in (full, rest):
        if o.evaluation is None:
            raise ValueError(f"{o.candidate_ids}: {o.reason}")
    return RemovalResult(
        objective, action_id, dependents,
        tuple(f"removing {action_id} also removes {d}" for d in dependents),
        full, rest, full.within_t - rest.within_t, rest.capped_time_sum - full.capped_time_sum, NON_ADDITIVE_NOTE,
    )

**Fixture catalog and candidates**

In [ ]:
FIXTURE = "fixture"


def fixture_service(sid, headway, km, speed, stops, dwell, layover, coverage="full"):
    return ServiceGroup(sid, (f"{sid}_out", f"{sid}_in"), headway, km, speed, stops, dwell, layover,
                        coverage, FIXTURE, FIXTURE)


catalog = ModelCatalog(
    service_groups={
        "SG_R2": fixture_service("SG_R2", 10, 12, 15, 30, 20, 5),
        "SG_R14": fixture_service("SG_R14", 12, None, None, None, None, None, coverage="partial"),
    },
    prebuilt_services={
        "SVC_SHUTTLE": fixture_service("SVC_SHUTTLE", 8, 9, 18, 6, 30, 4),
        "SVC_LOOP": fixture_service("SVC_LOOP", 20, 4, 20, 4, 30, 3),
    },
    links={
        "L1": Link("L1", "walk", 300, False, FIXTURE),
        "L2": Link("L2", "bike", 1200, True, FIXTURE),
        "L3": Link("L3", "walk", 40, True, FIXTURE),
    },
    parking={"P1": Parking("P1", FIXTURE)},
)

cands = {
    "a1": Candidate("a1", "headway_cut", target_id="SG_R2", pct=0.20),
    "a1b": Candidate("a1b", "headway_cut", target_id="SG_R2", pct=0.30),
    "a8": Candidate("a8", "headway_cut", target_id="SG_R2", pct=0.05),
    "a2": Candidate("a2", "headway_cut", target_id="SG_R14", pct=0.20),
    "a3": Candidate("a3", "activate_service", target_id="SVC_SHUTTLE"),
    "a3b": Candidate("a3b", "activate_service", target_id="SVC_SHUTTLE"),
    "a9": Candidate("a9", "activate_service", target_id="SVC_LOOP"),
    "a45": Candidate("a45", "link_bundle", link_ids=("L2",), parking_ids=("P1",)),
    "a6": Candidate("a6", "link_bundle", link_ids=("L1",)),
    "a7": Candidate("a7", "link_bundle", link_ids=("L2", "L3")),
}

base_scenario = load_scenario(Path("../config/limits.yaml"), speed_factor=1.0, headway_factor=1.0)


def res(*ids, scenario=base_scenario):
    return compute_resources([cands[i] for i in ids], catalog, scenario)


def veh(result, sid):
    s = next(s for s in result.per_service if s.service_id == sid)
    return s.base_vehicles, s.new_vehicles

**compute_resources tests**

In [ ]:
# Expected values computed by hand
r = res("a1");   assert (r.added_bus_hours, veh(r, "SG_R2"), r.new_infra_km, r.action_count, r.status, r.warnings) == (3, (7, 8), 0, 1, "known", ())  # E1
r = res("a1b");  assert (r.added_bus_hours, veh(r, "SG_R2")) == (6, (7, 9))                                     # E2
r = res("a8");   assert (r.added_bus_hours, veh(r, "SG_R2"), r.warnings) == (0, (7, 7), ("zero_cost_uses_slack",))  # E3
assert r.per_service[0].warnings == ("zero_cost_uses_slack",)
r = res("a3");   assert (r.added_bus_hours, veh(r, "SVC_SHUTTLE"), r.status) == (15, (0, 5), "known")           # E4
r = res("a9");   assert (r.added_bus_hours, veh(r, "SVC_LOOP"), r.warnings) == (3, (0, 1), ("headway_exceeds_cycle",))  # E5
assert r.per_service[0].warnings == ("headway_exceeds_cycle",)
r = res("a1", "a3", "a45")
assert r.added_bus_hours == 18 and math.isclose(r.new_infra_km, 1.2) and r.action_count == 3 and r.status == "known"  # E6
r = res("a45", "a7"); assert math.isclose(r.new_infra_km, 1.24) and r.action_count == 2                         # E7
r = res("a6");   assert (r.new_infra_km, r.action_count) == (0, 1)                                              # E8
r = res("a1", "a2"); assert (r.status, r.added_bus_hours, r.known_part_bus_hours) == ("unknown", None, 3)       # E9

try:                                                                                                             # E10
    res("a1", "a1b")
    raise AssertionError("E10: two cuts on one group accepted")
except ValueError as e:
    assert "a1" in str(e) and "a1b" in str(e)

try:                                                                                                             # same service activated twice
    res("a3", "a3b")
    raise AssertionError("two activations of one service accepted")
except ValueError as e:
    assert "a3" in str(e) and "a3b" in str(e)

r = res("a1", scenario=Scenario(0.8, 1.0, 3.0))                                                                 # E11
assert math.isclose(r.per_service[0].new_cycle_min, 75) and veh(r, "SG_R2") == (8, 10) and r.added_bus_hours == 6
r = res("a1", scenario=Scenario(1.0, 0.8, 3.0))                                                                 # E12
assert math.isclose(r.per_service[0].new_headway_min, 6.4) and veh(r, "SG_R2") == (8, 10) and r.added_bus_hours == 6
assert res("a3", "a1") == res("a1", "a3")                                                                       # E13

for bad in (lambda: Scenario(1.0, 1.0), lambda: Scenario(1.0, 1.0, None),
            lambda: Scenario(True, 1.0, 3.0), lambda: Scenario(1.0, True, 3.0), lambda: Scenario(1.0, 1.0, True)):                                      # E14
    try:
        bad()
        raise AssertionError("E14: scenario without H accepted")
    except (TypeError, ValueError):
        pass

assert vehicles(66.00000000000001, 6) == 11 and vehicles(66.06, 6) == 12 and vehicles(63, 10) == 7             # E15
for bad in (lambda: vehicles(63, 0), lambda: vehicles(63, -5),                                                  # E16
            *(lambda p=p: Candidate("bad", "headway_cut", target_id="SG_R2", pct=p) for p in (0, -0.1, 1, 1.5))):  # E17
    try:
        bad()
        raise AssertionError("E16/E17: bad input accepted")
    except ValueError:
        pass

print("compute_resources: all tests passed")

**Four-action fixture**

In [ ]:
fx = load_fixture(Path("../tests/fixtures/decision/fixture_4action.json"))
fx_evaluate = make_fixture_evaluator(fx)
fx_limits = fx["limits"]


def fx_request(ids):
    thresholds = tuple(sorted([fx_limits["T_min"], *fx_limits["T_sensitivity_min"]]))
    return RoutingRequest(tuple(ids), thresholds, fx_limits["T_min"], fx_limits["C_min"])


def fx_resources(ids):
    bus_hours = sum(fx["actions"][i]["bus_hours"] for i in ids)
    km = sum(fx["actions"][i]["new_infra_km"] for i in ids)
    return ResourceResult(bus_hours, bus_hours, km, len(ids), "known", (), ())

**Fixture evaluator and ranking tests**

In [ ]:
# Expected values computed by hand: n40 / n45 / n50 / capped sum
expected = {
    "": (0, 0, 2, 507), "A": (1, 2, 4, 467), "B": (2, 4, 6, 437), "C": (1, 3, 5, 398), "D": (0, 0, 2, 507),
    "AC": (2, 5, 7, 358), "AD": (1, 2, 4, 467), "BD": (2, 4, 6, 437), "CD": (1, 5, 5, 390), "ACD": (2, 7, 7, 350),
}
for pkg, (n40, n45, n50, capped) in expected.items():
    r = fx_evaluate(fx_request(pkg))
    assert (r.within_counts[40], r.within_counts[45], r.within_counts[50], r.capped_time_sum) == (n40, n45, n50, capped), pkg

assert [od.minutes for od in fx_evaluate(fx_request("ACD")).od_results] == [40, 42, 55, 41, 39, 44, 44, 45]
od5 = fx_evaluate(fx_request("B")).od_results[4]
assert (od5.od_id, od5.minutes, od5.status) == ("OD5", 120, "no_path")

ranked = sorted(expected, key=lambda p: rank_key(fx_evaluate(fx_request(p)), fx_resources(p), objective="count"))
assert ranked == ["ACD", "AC", "CD", "B", "BD", "C", "A", "AD", "", "D"], ranked

synthetic = EvaluationResult("synthetic", ("X",), (), {45: 5}, 45, 300)
assert rank_key(synthetic, ResourceResult(6, 6, 0, 1, "known", (), ()), objective="count") < rank_key(synthetic, ResourceResult(9, 9, 0, 1, "known", (), ()), objective="count")
assert rank_key(synthetic, ResourceResult(6, 6, 0.5, 1, "known", (), ()), objective="count") < rank_key(synthetic, ResourceResult(6, 6, 0.5, 2, "known", (), ()), objective="count")

print("fixture evaluator and rank_key: all tests passed")

**Fixture decision problem**

In [ ]:
import tempfile


def fx_table_resources(table):
    """Fixture-only resources: summed from the action table (not used in production)."""
    def resources(ids):
        ids = set(ids)
        bus_hours = sum(table[i]["bus_hours"] for i in ids)
        km = sum(table[i]["new_infra_km"] for i in ids)
        return ResourceResult(bus_hours, bus_hours, km, len(ids), "known", (), ())
    return resources


def fx_problem(objective, max_exact=1024):
    """Objective set explicitly for the fixture; the gate is not run on synthetic data."""
    path = Path(tempfile.mkdtemp()) / "objective_decision.json"
    path.write_text(json.dumps({"objective": objective, "note": "fixture: set explicitly, gate not run"}))
    return DecisionProblem(
        candidate_ids=tuple(sorted(fx["actions"])),
        exclusions=tuple(tuple(g) for g in fx["exclusions"]),
        prerequisites={},
        distance_m={a: v["distance_m"] for a, v in fx["actions"].items()},
        limits=Limits(fx_limits["B_h_bus_hours"], fx_limits["K_new_infra_km"], fx_limits["N_a_actions"], max_exact),
        thresholds_min=(40, 45, 50),
        primary_t_min=fx_limits["T_min"],
        cap_min=fx_limits["C_min"],
        objective_path=path,
        resources=fx_table_resources(fx["actions"]),
    )


def show(result):
    rec = result.recommendation
    print(f"{result.objective} / {result.label}: {''.join(rec.candidate_ids) or '-'} "
          f"({rec.within_t} ODs, sum {rec.capped_time_sum:g}) from {result.recommendation_source}")
    for a in result.alternatives:
        print(f"  alt {''.join(a.candidate_ids) or '-'}: {a.reason}")
    for name, b in result.baselines.items():
        print(f"  {b.label}: {''.join(b.package.candidate_ids) or '-'} "
              f"({b.package.within_t}, {b.package.capped_time_sum:g})  {[(s.candidate_id, s.outcome) for s in b.steps]}")

**Optimisation tests on the fixture**

In [ ]:
# Expected values computed by hand
r = optimize(fx_problem("count"), fx_evaluate)
show(r)
assert (r.label, r.recommendation.candidate_ids, r.recommendation.within_t, r.recommendation.capped_time_sum) == ("exact", ("A", "C", "D"), 7, 350)
assert r.feasible_count == 10
assert r.baselines["greedy"].package.candidate_ids == ("B",)
assert r.baselines["nearest_gap"].package.candidate_ids == ("A", "C", "D")
assert [a.candidate_ids for a in r.alternatives] == [("A", "C"), ("C", "D"), ("B",)]
assert [a.reason for a in r.alternatives] == ["2 fewer ODs within 45 min"] * 2 + ["3 fewer ODs within 45 min"]

r = optimize(fx_problem("sum"), fx_evaluate)
show(r)
assert r.recommendation.candidate_ids == ("A", "C", "D") and r.baselines["greedy"].package.candidate_ids == ("A", "C")

r = optimize(fx_problem("count", max_exact=5), fx_evaluate)
show(r)
assert (r.label, r.recommendation.candidate_ids, r.recommendation_source) == ("approximate", ("A", "C", "D"), "nearest_gap")
assert r.approximate_reason == "catalog too large for exact search (12 > 5)"

assert gate_objective({40: 60, 45: 75, 50: 90}, 500)[0] == "count"
assert gate_objective({40: 8, 45: 8, 50: 9}, 500)[0] == "sum"

print("optimize: all tests passed")

**Stress fixture (synthetic time model)**

In [ ]:
import copy

sx = load_fixture(Path("../tests/fixtures/decision/fixture_stress.json"))
sx_lim = sx["limits"]
sx_default = Scenario(1.0, 1.0, sx_lim["H_hours"])


def sx_service(sid, v):
    return ServiceGroup(sid, (), v["base_headway_min"], v["full_cycle_km"], v["speed_kmh"], v["stops_per_cycle"],
                        v["dwell_s"], v["layover_min"], v["coverage"], "fixture", "fixture")


sx_catalog = ModelCatalog(
    service_groups={k: sx_service(k, v) for k, v in sx["catalog"]["service_groups"].items()},
    prebuilt_services={k: sx_service(k, v) for k, v in sx["catalog"]["prebuilt_services"].items()},
    links={k: Link(k, v["kind"], v["length_m"], v["requires_construction"], "fixture") for k, v in sx["catalog"]["links"].items()},
    parking={},
)
sx_candidates = {c: Candidate(c, a["kind"], target_id=a.get("target_id"), pct=a.get("pct"), link_ids=tuple(a.get("link_ids", ())))
                 for c, a in sx["actions"].items()}


def sx_evaluator_for(scenario):
    """Synthetic fixture time model: t' = walk + h'/2 + (t - walk - h0/2) / speed_factor."""
    def headway(rides, requires, sc):
        if rides in sx["catalog"]["prebuilt_services"]:
            return sx["catalog"]["prebuilt_services"][rides]["base_headway_min"]
        group, kind = rides.split(":")
        h = sx["catalog"]["service_groups"][group]["base_headway_min"] * sc.group_headway_factor(group)
        cut = [sx["actions"][c]["pct"] for c in requires if sx["actions"][c].get("target_id") == group]
        return h * (1 - cut[0]) if kind == "new" else h

    def at(t, rides, requires):
        h0, h1 = headway(rides, requires, sx_default), headway(rides, requires, scenario)
        return sx["walk_min"] + h1 / 2 + (t - sx["walk_min"] - h0 / 2) / scenario.speed_factor

    moved = copy.deepcopy(sx)
    for od in moved["ods"]:
        if od["disrupted_min"] is not None:
            od["disrupted_min"] = at(od["disrupted_min"], od["disrupted_rides"], ())
        for opt in od["options"]:
            opt["minutes"] = at(opt["minutes"], opt["rides"], opt["requires"])
    return make_fixture_evaluator(moved)


sx_objective = Path(tempfile.mkdtemp()) / "objective_decision.json"
sx_objective.write_text(json.dumps({"objective": sx["objective"], "note": "fixture: set explicitly, gate not run"}))
sx_template = DecisionProblem(
    candidate_ids=tuple(sorted(sx_candidates)),
    exclusions=tuple(tuple(g) for g in sx["exclusions"]),
    prerequisites={},
    distance_m={c: a["distance_m"] for c, a in sx["actions"].items()},
    limits=Limits(sx_lim["B_h_bus_hours"], sx_lim["K_new_infra_km"], sx_lim["N_a_actions"], sx_lim["max_exact_packages"]),
    thresholds_min=(40, 45, 50),
    primary_t_min=sx_lim["T_min"],
    cap_min=sx_lim["C_min"],
    objective_path=sx_objective,
    resources=make_catalog_resources(sx_candidates, sx_catalog, sx_default),
)
sx_setup = StressSetup(make_problem_for(sx_template, sx_candidates, sx_catalog), sx_evaluator_for,
                       StressGrid(sx["grid"]["speed_levels_kmh"], sx["grid"]["headway_levels_min"]), sx_lim["H_hours"])

**Stress and removal checks on the fixture**

In [ ]:
# Expected values computed by hand
rec = optimize(sx_setup.problem_for(sx_default), sx_evaluator_for(sx_default)).recommendation
assert (rec.candidate_ids, rec.within_t, rec.capped_time_sum, rec.resources.added_bus_hours) == (("A", "C", "D"), 7, 350.0, 24)

st = stress_test(sx_setup, ("A", "C", "D"))
print(f"objective {st.objective}; package {'+'.join(st.package_ids)}")
print("speed    headway  feasible  bus-h  overrun  ODs<=45  capped")
for p in st.points:
    print(f"{p.speed_level:8} {p.headway_level:8} {str(p.feasible):9} {p.bus_hours:5g}  {p.overrun:7g}  {p.within_t:7}  {p.capped_time_sum:.2f}")
print(f"worst: {st.worst.speed_level}/{st.worst.headway_level}, overrun {st.worst.overrun:g}")
r = st.reoptimization.recommendation
print(f"re-optimisation: {'+'.join(r.candidate_ids)} ({r.within_t} ODs, sum {r.capped_time_sum:.2f}, "
      f"{r.resources.added_bus_hours:g} bus-hours); {st.reoptimization_reason}")
print("warnings:", st.warnings or "none")
assert [p.bus_hours for p in st.points] == [33, 30, 24, 24, 24, 21, 21, 18, 15]
assert [p.within_t for p in st.points] == [0, 0, 0, 7, 7, 7, 8, 8, 8]
assert r.candidate_ids == ("C", "D") and abs(r.capped_time_sum - 535.13) <= 0.01

four_problem = fx_problem("count")
for action in ("A", "C", "D"):
    rm = remove_action(four_problem, ("A", "C", "D"), action, fx_evaluate)
    print(f"remove {action} -> {''.join(rm.remaining.candidate_ids)}: {rm.remaining.within_t} ODs, "
          f"sum {rm.remaining.capped_time_sum:g}, loss {rm.loss_within_t}")
print("note:", rm.note)
print("stress and removal: all tests passed")

**Bus data curation**

**Paths and constants (phase2 JSON is byte-identical to phase1)**

In [ ]:
import csv

BUS_RAW = Path("../data/raw/ayna_bus/phase2")
BUS_RAW_P1 = Path("../data/raw/ayna_bus/phase1")
BUS_OUT = Path("../data/curated")
BUS_DIRECTIONS = {1: "A", 2: "B"}  # directionTypeId -> direction
SNAP_LIMIT_M = 50
EARTH_R_M = 6371008.8
API_URL = "https://map-api.ayna.gov.az/api/bus/getBusById?id={}"

**Geometry helpers (local equirectangular metres, one origin per route)**

In [ ]:
def make_projector(points):
    """(lat, lon) -> (x, y) metres around the mean point; all lengths use this plane."""
    lat0 = sum(p[0] for p in points) / len(points)
    lon0 = sum(p[1] for p in points) / len(points)
    ky = math.radians(1) * EARTH_R_M
    kx = ky * math.cos(math.radians(lat0))
    return lambda lat, lon: ((lon - lon0) * kx, (lat - lat0) * ky)


def cumulative_m(xy):
    cum = [0.0]
    for (x0, y0), (x1, y1) in zip(xy, xy[1:]):
        cum.append(cum[-1] + math.hypot(x1 - x0, y1 - y0))
    return cum


def project_on_polyline(pt, xy, cum):
    """Nearest point on the polyline: (chainage_m, snap_m)."""
    best = None
    for i in range(len(xy) - 1):
        (x0, y0), (x1, y1) = xy[i], xy[i + 1]
        dx, dy = x1 - x0, y1 - y0
        seg2 = dx * dx + dy * dy
        t = 0.0 if seg2 == 0 else max(0.0, min(1.0, ((pt[0] - x0) * dx + (pt[1] - y0) * dy) / seg2))
        snap = math.hypot(pt[0] - (x0 + t * dx), pt[1] - (y0 + t * dy))
        if best is None or snap < best[1]:
            best = (cum[i] + t * math.sqrt(seg2), snap)
    return best

**Load one AYNA route JSON (data only, fail loudly)**

In [ ]:
def load_route(path):
    d = json.loads(Path(path).read_text(encoding="utf-8"))
    for key in ("id", "number", "routLength", "durationMinuts", "stops", "routes"):
        if d.get(key) is None:
            raise ValueError(f"{path}: missing {key}")
    polylines, stops = {}, {"A": [], "B": []}
    for r in d["routes"]:
        direction = BUS_DIRECTIONS[r["directionTypeId"]]
        if direction in polylines:
            raise ValueError(f"{path}: two polylines for direction {direction}")
        polylines[direction] = [(c["lat"], c["lng"]) for c in r["flowCoordinates"]]
    if set(polylines) != {"A", "B"}:
        raise ValueError(f"{path}: need polylines for A and B, got {sorted(polylines)}")
    for s in d["stops"]:
        if s["stop"]["id"] != s["stopId"]:
            raise ValueError(f"{path}: stopId {s['stopId']} != nested stop id {s['stop']['id']}")
        stops[BUS_DIRECTIONS[s["directionTypeId"]]].append({
            "stop_id": str(s["stopId"]), "name": s["stopName"],
            "lat": float(s["stop"]["latitude"]), "lon": float(s["stop"]["longitude"]),
        })
    return {"route": str(d["number"]), "api_id": d["id"], "routLength_km": float(d["routLength"]),
            "duration_min": float(d["durationMinuts"]), "polylines": polylines, "stops": stops,
            "file": Path(path).name}


def check_against_stops_csv(rd, csv_path):
    """Stop order and coordinates in the JSON must match the curated-by-hand stops CSV."""
    with open(csv_path, newline="", encoding="utf-8-sig") as f:
        rows = list(csv.DictReader(f))
    for direction in ("A", "B"):
        ref = sorted((r for r in rows if r["direction"] == direction), key=lambda r: int(r["seq"]))
        got = rd["stops"][direction]
        if len(ref) != len(got) or any(
            (float(r["lat"]), float(r["lon"])) != (g["lat"], g["lon"]) for r, g in zip(ref, got)
        ):
            raise ValueError(f"route {rd['route']} {direction}: JSON stops differ from {csv_path}")

**Patterns: chainage, segments and snap per stop**

In [ ]:
def build_patterns(rd):
    """Pattern rows and geometry for one route; flags are written to notes, nothing is moved or dropped."""
    pts = [p for pl in rd["polylines"].values() for p in pl]
    proj = make_projector(pts)
    rows, geo = [], {}
    for direction in ("A", "B"):
        xy = [proj(*p) for p in rd["polylines"][direction]]
        cum = cumulative_m(xy)
        geo[direction] = {"coords": [(lon, lat) for lat, lon in rd["polylines"][direction]], "length_m": cum[-1]}
        part = []
        for seq, s in enumerate(rd["stops"][direction], 1):
            chain, snap = project_on_polyline(proj(s["lat"], s["lon"]), xy, cum)
            notes = []
            if snap > SNAP_LIMIT_M:
                notes.append(f"snap_m>{SNAP_LIMIT_M}")
            if part and chain <= part[-1]["chainage_m"]:
                notes.append("non_increasing_chainage")
            part.append({
                "route": rd["route"], "direction": direction, "pattern_id": f"{rd['route']}_{direction}",
                "seq": seq, "stop_id": s["stop_id"], "chainage_m": chain, "segment_m": None, "snap_m": snap,
                "source": f"ayna_map {API_URL.format(rd['api_id'])} | evidence/{rd['file']}",
                "notes": "; ".join(notes), "_name": s["name"], "_lat": s["lat"], "_lon": s["lon"], "_api_id": rd["api_id"],
            })
        for a, b in zip(part, part[1:]):
            a["segment_m"] = b["chainage_m"] - a["chainage_m"]
        rows += part
    return rows, geo

**Stops: one row per stop id, suffix only when coordinates conflict**

In [ ]:
def curate_stops(pattern_rows):
    """Replace raw stop ids in pattern rows by bus_stops keys and return bus_stops rows."""
    coords = {}
    for r in pattern_rows:
        coords.setdefault(r["stop_id"], set()).add((r["_lat"], r["_lon"]))
    conflicting = {k for k, v in coords.items() if len(v) > 1}
    stops = {}
    for r in pattern_rows:
        raw = r["stop_id"]
        key = f"{raw}_{r['route']}{r['direction']}" if raw in conflicting else raw
        r["stop_id"] = key
        s = stops.setdefault(key, {"stop_id": key, "names": [], "lat": r["_lat"], "lon": r["_lon"], "ev": [], "conflict": raw in conflicting})
        if r["_name"] not in s["names"]:
            s["names"].append(r["_name"])
        ev = f"{API_URL.format(r['_api_id'])} | evidence/ayna_route_{r['route']}.json"
        if ev not in s["ev"]:
            s["ev"].append(ev)
    out = []
    for s in sorted(stops.values(), key=lambda s: (int(s["stop_id"].split("_")[0]), s["stop_id"])):
        notes = []
        if s["conflict"]:
            notes.append("same AYNA stopId has different coordinates per direction; kept as separate rows with route+direction suffix")
        if len(s["names"]) > 1:
            notes.append("name differs between directions/routes: " + " / ".join(s["names"]))
        out.append({"stop_id": s["stop_id"], "name": s["names"][0], "lat": s["lat"], "lon": s["lon"],
                    "source": "ayna_map", "evidence": " ; ".join(s["ev"]), "notes": "; ".join(notes)})
    return out

**Assumptions P4/P5 and service facts (fail loudly if absent)**

In [ ]:
def read_rows(path):
    with open(path, newline="", encoding="utf-8-sig") as f:
        return list(csv.DictReader(f))


def load_dwell_layover(path=BUS_RAW_P1 / "templates" / "assumptions.csv"):
    by_id = {r["param_id"]: r for r in read_rows(path)}
    p4, p5 = by_id["P4"], by_id["P5"]
    if p4["unit"] != "saniyə" or p5["unit"] != "min":
        raise ValueError(f"unexpected P4/P5 units: {p4['unit']!r}, {p5['unit']!r}")
    return {"dwell_s": float(p4["value_default"]), "layover_per_terminal_min": float(p5["value_default"]),
            "status": p4["status"], "layover_status": p5["status"]}


def fact(facts, route, metric):
    rows = [r for r in facts if r["route"] == route and r["metric"] == metric]
    if len(rows) != 1:
        raise ValueError(f"route {route}: need exactly one {metric!r} row in route_service_facts.csv, got {len(rows)}")
    return rows[0]

**Curated assumptions: P1/P2 headways replace the first-round estimate**

In [ ]:
# P1/P2 decision (Ali, 2026-10-17); pending a live MaaS count. The raw file is never edited.
HEADWAY_UPDATES = {
    "P1": {"value_low": "6", "value_default": "7.5", "value_high": "10", "status": "published",
           "fact": ("2", "headway_operator_reported"),
           "justification": "Oxu.az 2026-08-31 (BakuBus: 24 buses, 7-8 min); cross-checked with full cycle 2*73+10 min / 24 buses = 6.5 min"},
    "P2": {"value_low": "4.5", "value_default": "5", "value_high": "8", "status": "assumption",
           "fact": ("14", "vehicles_required_tender"),
           "justification": "AYNA tender 2025-10-01 lot 333 (20 buses) / full cycle 2*45+10 min"},
}


def curate_assumptions(raw_path, facts, out_path, extra_paths=()):
    """Copy raw assumptions.csv (plus extra files, values unchanged), replacing P1/P2 and keeping the previous values in notes."""
    raw = read_rows(raw_path)
    for extra in extra_paths:
        rows = read_rows(extra)
        if not rows or list(rows[0]) != list(raw[0]):
            raise ValueError(f"{extra}: columns differ from {raw_path}")
        raw += rows
    ids = [r["param_id"] for r in raw]
    if len(ids) != len(set(ids)):
        raise ValueError(f"duplicate param_id across assumption files: {sorted({i for i in ids if ids.count(i) > 1})}")
    by_id = {r["param_id"]: r for r in raw}
    missing = set(HEADWAY_UPDATES) - set(by_id)
    if missing:
        raise ValueError(f"{raw_path}: missing {sorted(missing)}")
    rep, buses = fact(facts, "2", "headway_operator_reported"), fact(facts, "2", "vehicles_operator_reported")
    if rep["value"] != "7-8" or buses["value"] != "24":
        raise ValueError("route 2: raw facts no longer match the Oxu.az headway basis")
    if fact(facts, "14", "vehicles_required_tender")["value"] != "20":
        raise ValueError("route 14: tender vehicle count differs from the 20 buses in the basis")
    out = []
    for r in raw:
        row = dict(r, notes="")
        update = HEADWAY_UPDATES.get(r["param_id"])
        if update:
            src = fact(facts, *update["fact"])
            row.update({k: update[k] for k in ("value_low", "value_default", "value_high", "status", "justification")})
            row.update(source_url=src["source_url"], source_date=src["source_date"], filled_by="Ali")
            row["notes"] = (f"superseded: {r['value_low']}/{r['value_default']}/{r['value_high']}, first-round estimate "
                            f"without sources (previous status {r['status']}, filled_by {r['filled_by']}: {r['justification']})")
        out.append(row)
    write_csv(out_path, [*raw[0].keys(), "notes"], out)
    return out


def headway_for(route, assumptions):
    rows = [r for r in assumptions if r["applies_to"] == f"route_{route}_peak"]
    if len(rows) != 1:
        raise ValueError(f"route {route}: need exactly one route_{route}_peak row in curated assumptions.csv, got {len(rows)}")
    r = rows[0]
    source = (f"{r['param_id']} in data/curated/assumptions.csv: {r['justification']}; {r['source_url']} "
              f"({r['source_date']}); pending a live MaaS count")
    return float(r["value_low"]), float(r["value_default"]), float(r["value_high"]), r["status"], source

**Service group per route (speed from the full cycle, geometry basis)**

In [ ]:
def build_service(rd, pattern_rows, geo, facts, dl, assumptions):
    route = rd["route"]
    intermediate = sum(len(rd["stops"][d]) - 2 for d in ("A", "B"))  # terminals excluded
    cycle_km = sum(g["length_m"] for g in geo.values()) / 1000
    duration = rd["duration_min"]
    dur_fact = fact(facts, route, "durationMinuts_raw")
    if dur_fact["status"] != "published" or float(dur_fact["value"]) != duration:
        raise ValueError(f"route {route}: duration in JSON and raw facts disagree")
    moving_min = 2 * duration - intermediate * dl["dwell_s"] / 60
    if moving_min <= 0:
        raise ValueError(f"route {route}: no moving time left after dwell")
    speed = 60 * cycle_km / moving_min
    low, default, high, hw_status, hw_source = headway_for(route, assumptions)
    flagged = [f"{r['pattern_id']}#{r['seq']}" for r in pattern_rows if r["notes"]]
    notes = [
        "assumption: durationMinuts is the mean one-way duration (raw facts: meaning unconfirmed)",
        "distance basis is the geometry; routLength is a cross-check only",
        "headways read from data/curated/assumptions.csv (P1/P2); raw first-round 4/6/10 superseded",
        "layover_min = P5 per terminal x 2 terminals = full-cycle sum",
    ]
    if flagged:
        notes.append("flagged stops: " + ", ".join(flagged))
    return {
        "service_group_id": f"SG_R{route}", "route": route,
        "pattern_ids": ";".join(f"{route}_{d}" for d in ("A", "B")),
        "cycle_km_geometry": cycle_km, "routLength_km": rd["routLength_km"],
        "length_diff_pct": 100 * (cycle_km - rd["routLength_km"]) / rd["routLength_km"],
        "intermediate_stops": intermediate, "duration_min": duration, "duration_status": dur_fact["status"],
        "speed_kmh": speed, "speed_status": "assumption",
        "speed_method": "60*cycle_km_geometry/(2*duration_min - intermediate_stops*dwell_s/60); layover and passenger waiting excluded",
        "dwell_s": dl["dwell_s"], "dwell_status": dl["status"],
        "layover_min": 2 * dl["layover_per_terminal_min"], "layover_status": dl["layover_status"],
        "headway_low_min": low, "headway_default_min": default, "headway_high_min": high,
        "headway_status": hw_status, "headway_source": hw_source,
        "coverage": "full", "notes": "; ".join(notes),
    }

**Run curation over every route JSON and write data/curated**

In [ ]:
STOP_FIELDS = ["stop_id", "name", "lat", "lon", "source", "evidence", "notes"]
PATTERN_FIELDS = ["route", "direction", "pattern_id", "seq", "stop_id", "chainage_m", "segment_m", "snap_m", "source", "notes"]
SERVICE_FIELDS = [
    "service_group_id", "route", "pattern_ids", "cycle_km_geometry", "routLength_km", "length_diff_pct",
    "intermediate_stops", "duration_min", "duration_status", "speed_kmh", "speed_status", "speed_method",
    "dwell_s", "dwell_status", "layover_min", "layover_status", "headway_low_min", "headway_default_min",
    "headway_high_min", "headway_status", "headway_source", "coverage", "notes",
]


def write_csv(path, fields, rows):
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=fields, extrasaction="ignore")
        w.writeheader()
        for r in rows:
            w.writerow({k: ("" if r[k] is None else r[k]) for k in fields})


def curate_buses():
    files = sorted(BUS_RAW.glob("evidence/ayna_route_*.json"))
    if not files:
        raise FileNotFoundError(f"no route JSON in {BUS_RAW}")
    facts = read_rows(BUS_RAW / "templates" / "route_service_facts.csv")
    BUS_OUT.mkdir(parents=True, exist_ok=True)
    curate_assumptions(BUS_RAW_P1 / "templates" / "assumptions.csv", facts, BUS_OUT / "assumptions.csv",
                       extra_paths=[BUS_RAW / "templates" / "assumptions_P16_P18.csv"])
    assumptions = read_rows(BUS_OUT / "assumptions.csv")
    dl = load_dwell_layover(BUS_OUT / "assumptions.csv")
    if (dl["dwell_s"], dl["layover_per_terminal_min"]) != (25.0, 5.0):
        raise ValueError(f"P4/P5 defaults differ from the agreed 25 s / 5 min: {dl}")
    all_rows, services, features, per_route = [], [], [], {}
    for path in files:
        rd = load_route(path)
        check_against_stops_csv(rd, BUS_RAW_P1 / "templates" / f"stops_route_{rd['route']}.csv")
        rows, geo = build_patterns(rd)
        per_route[rd["route"]] = (rd, rows, geo)
        all_rows += rows
        for d, g in geo.items():
            features.append({"type": "Feature", "geometry": {"type": "LineString", "coordinates": g["coords"]},
                             "properties": {"route": rd["route"], "direction": d, "pattern_id": f"{rd['route']}_{d}",
                                            "length_m": g["length_m"], "source": f"ayna_map flowCoordinates, evidence/{rd['file']}"}})
    stop_rows = curate_stops(all_rows)  # rewrites stop_id in all_rows
    for rd, rows, geo in per_route.values():
        services.append(build_service(rd, rows, geo, facts, dl, assumptions))
    write_csv(BUS_OUT / "bus_stops.csv", STOP_FIELDS, stop_rows)
    write_csv(BUS_OUT / "bus_patterns.csv", PATTERN_FIELDS, all_rows)
    write_csv(BUS_OUT / "bus_services.csv", SERVICE_FIELDS, services)
    (BUS_OUT / "bus_route_geometry.geojson").write_text(
        json.dumps({"type": "FeatureCollection", "features": features}, ensure_ascii=False), encoding="utf-8")
    return stop_rows, all_rows, services


bus_stop_rows, bus_pattern_rows, bus_service_rows = curate_buses()
print(len(bus_stop_rows), "stops;", len(bus_pattern_rows), "pattern rows;", len(bus_service_rows), "services")

**Curation check on synthetic input (conflicting stop ids get a suffix)**

In [ ]:
def _row(route, direction, sid, lat):
    return {"route": route, "direction": direction, "stop_id": sid, "_lat": lat, "_lon": 49.0,
            "_name": "X", "_api_id": 1}


_rows = [_row("2", "A", "7", 40.0), _row("2", "B", "7", 40.0), _row("2", "A", "8", 40.1), _row("2", "B", "8", 40.2)]
_out = curate_stops(_rows)
assert [r["stop_id"] for r in _out] == ["7", "8_2A", "8_2B"]
assert [r["stop_id"] for r in _rows] == ["7", "7", "8_2A", "8_2B"]
assert "different coordinates per direction" in _out[1]["notes"]
_xy = [(0.0, 0.0), (100.0, 0.0), (100.0, 100.0)]
assert project_on_polyline((50.0, 10.0), _xy, cumulative_m(_xy)) == (50.0, 10.0)
assert project_on_polyline((120.0, 40.0), _xy, cumulative_m(_xy)) == (140.0, 20.0)

**Per-route summary and flagged stops**

In [ ]:
print("route  geom_km  routLength_km  diff_%   stops(A+B)  speed_kmh(geometry)  speed_kmh(routLength)")
for s in bus_service_rows:
    n_stops = sum(1 for r in bus_pattern_rows if r["route"] == s["route"])
    moving = 2 * s["duration_min"] - s["intermediate_stops"] * s["dwell_s"] / 60
    print(f"{s['route']:>5}  {s['cycle_km_geometry']:.3f}  {s['routLength_km']:>13}  {s['length_diff_pct']:+.2f}  "
          f"{n_stops:>10}  {s['speed_kmh']:>19.6f}  {60 * s['routLength_km'] / moving:>21.6f}")
flagged = [r for r in bus_pattern_rows if r["notes"]]
print("flagged stops:", len(flagged))
for r in flagged:
    print(r["pattern_id"], r["seq"], r["stop_id"], f"chainage={r['chainage_m']:.1f} snap={r['snap_m']:.1f}", r["notes"])